# selman bağımsız kontrol

egenin notebookundan bağımsız olarak caltech verisini okuyup sayıları ve doluluk saatlerini kontrol ediyorum
model eğitmiyorum sadece doğrulama

## 1 dosyaları oku

In [1]:
from pathlib import Path
import json

import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "raw" / "acn"
json_files = sorted(data_dir.glob("caltech_*.json"))

assert len(json_files) == 14, f"14 dosya olmali ama {len(json_files)} tane var"
print("dosya sayisi", len(json_files))
for path in json_files:
    print(path.name)

dosya sayisi 14
caltech_2018_Q2.json
caltech_2018_Q3.json
caltech_2018_Q4.json
caltech_2019_Q1.json
caltech_2019_Q2.json
caltech_2019_Q3.json
caltech_2019_Q4.json
caltech_2020_Q1.json
caltech_2020_Q2.json
caltech_2020_Q3.json
caltech_2020_Q4.json
caltech_2021_Q1.json
caltech_2021_Q2.json
caltech_2021_Q3.json


In [2]:
expected_counts = {
    "caltech_2018_Q2.json": 3880,
    "caltech_2018_Q3.json": 6793,
    "caltech_2018_Q4.json": 4624,
    "caltech_2019_Q1.json": 2876,
    "caltech_2019_Q2.json": 2884,
    "caltech_2019_Q3.json": 2508,
    "caltech_2019_Q4.json": 2349,
    "caltech_2020_Q1.json": 2051,
    "caltech_2020_Q2.json": 81,
    "caltech_2020_Q3.json": 71,
    "caltech_2020_Q4.json": 269,
    "caltech_2021_Q1.json": 614,
    "caltech_2021_Q2.json": 1085,
    "caltech_2021_Q3.json": 1339,
}

all_sessions = []
per_file_counts = {}

for file_path in json_files:
    with file_path.open(encoding="utf-8") as file:
        quarter_data = json.load(file)

    quarter_sessions = quarter_data["_items"]
    per_file_counts[file_path.name] = len(quarter_sessions)

    expected = expected_counts[file_path.name]
    assert len(quarter_sessions) == expected, (
        f"{file_path.name}: {len(quarter_sessions)} != {expected}"
    )

    for session in quarter_sessions:
        session = dict(session)
        session["_source_file"] = file_path.name
        all_sessions.append(session)

sessions = pd.DataFrame(all_sessions)
print("satir sutun", sessions.shape)
print("oturum sayisi", len(sessions))
print("benzersiz sessionID", sessions["sessionID"].nunique())
print("sarj noktasi sayisi", sessions["stationID"].nunique())

assert len(sessions) == 31424
assert sessions["sessionID"].nunique() == 31424
assert sessions["stationID"].nunique() == 55
print("dosya basina sayilar egenin listesiyle uyumlu")

satir sutun (31424, 14)
oturum sayisi 31424
benzersiz sessionID 31424
sarj noktasi sayisi 55
dosya basina sayilar egenin listesiyle uyumlu


## 2 tarih kalitesi

In [3]:
time_columns = ["connectionTime", "disconnectTime", "doneChargingTime"]

for column in time_columns:
    sessions[f"{column}_utc"] = pd.to_datetime(
        sessions[column], utc=True, errors="coerce"
    )

missing_times = sessions[["connectionTime_utc", "disconnectTime_utc"]].isna().any(axis=1)
invalid_intervals = sessions["disconnectTime_utc"] <= sessions["connectionTime_utc"]
duplicate_ids = sessions["sessionID"].duplicated()

print("eksik veya okunamayan tarih", int(missing_times.sum()))
print("hatali zaman araligi", int(invalid_intervals.sum()))
print("tekrar eden oturum", int(duplicate_ids.sum()))

assert not missing_times.any()
assert not invalid_intervals.any()
assert not duplicate_ids.any()

sessions = sessions.sort_values("connectionTime_utc").reset_index(drop=True)
print("ilk baglanti utc", sessions["connectionTime_utc"].min())
print("son baglanti utc", sessions["connectionTime_utc"].max())

eksik veya okunamayan tarih 0
hatali zaman araligi 0
tekrar eden oturum 0
ilk baglanti utc 2018-04-25 11:08:04+00:00
son baglanti utc 2021-09-14 01:52:37+00:00


## 3 islenmis csv ile kiyas

eger egenin kaydettigi caltech_all_sessions.csv varsa sessionID kumesini karsilastiriyorum

In [4]:
processed_path = project_dir / "data" / "processed" / "caltech_all_sessions.csv"

if processed_path.exists():
    processed = pd.read_csv(processed_path)
    print("processed satir", len(processed))
    mine = set(sessions["sessionID"])
    theirs = set(processed["sessionID"].astype(str))
    print("sadece bende", len(mine - theirs))
    print("sadece processedda", len(theirs - mine))
    assert mine == theirs
    print("sessionID kumesı ayni")
else:
    print("processed csv yok atlaniyor", processed_path)

processed satir 31424
sadece bende 0
sadece processedda 0
sessionID kumesı ayni


## 4 saatlik doluluk matrisi

kural egeninkiyle ayni: baglanti ceil h ile baslar ayrilma ceil h den onceki saat baslarina 1 yazilir doneChargingTime kullanilmaz

In [5]:
def occupied_hours_for_session(connection_utc, disconnect_utc):
    hour = connection_utc.ceil("h")
    end_hour = disconnect_utc.ceil("h")
    hours = []
    while hour < end_hour:
        hours.append(hour)
        hour = hour + pd.Timedelta(hours=1)
    return hours


occupied_rows = []
for session in sessions.itertuples():
    for hour in occupied_hours_for_session(
        session.connectionTime_utc, session.disconnectTime_utc
    ):
        occupied_rows.append(
            {
                "hour_utc": hour,
                "stationID": session.stationID,
                "occupied": 1,
            }
        )

occupied_hours = pd.DataFrame(occupied_rows)
occupancy_matrix = occupied_hours.pivot_table(
    index="hour_utc",
    columns="stationID",
    values="occupied",
    aggfunc="max",
)

all_hours = pd.date_range(
    start=sessions["connectionTime_utc"].min().floor("h"),
    end=sessions["disconnectTime_utc"].max().ceil("h"),
    freq="h",
    inclusive="left",
)
station_ids = sorted(sessions["stationID"].dropna().unique())
occupancy_matrix = occupancy_matrix.reindex(index=all_hours, columns=station_ids)
occupancy_matrix = occupancy_matrix.fillna(0).astype(int)

print("saatlik tablo", occupancy_matrix.shape)
print("beklenen yaklasik", (29705, 55))
print("ilk saat", occupancy_matrix.index.min())
print("son saat", occupancy_matrix.index.max())
print(
    "en az bir nokta dolu olan saat",
    int((occupancy_matrix.sum(axis=1) > 0).sum()),
)

assert occupancy_matrix.shape[1] == 55
assert occupancy_matrix.shape[0] == 29705, occupancy_matrix.shape
print("matris boyutu egenin raporuyla uyumlu")

saatlik tablo (29705, 55)
beklenen yaklasik (29705, 55)
ilk saat 2018-04-25 11:00:00+00:00
son saat 2021-09-14 03:00:00+00:00
en az bir nokta dolu olan saat 19791
matris boyutu egenin raporuyla uyumlu


## 5 elle oturum kontrolu

uc oturumu ham zamandan beklenen saat listesiyle matrise bakiyorum

In [6]:
def manual_check(session_id):
    row = sessions.loc[sessions["sessionID"] == session_id].iloc[0]
    expected = occupied_hours_for_session(
        row["connectionTime_utc"], row["disconnectTime_utc"]
    )
    station = row["stationID"]

    print("sessionID", session_id)
    print("stationID", station)
    print("connection", row["connectionTime_utc"])
    print("disconnect", row["disconnectTime_utc"])
    print("beklenen dolu saat sayisi", len(expected))
    print("beklenen ilk/son", expected[0] if expected else None, expected[-1] if expected else None)

    for hour in expected:
        value = int(occupancy_matrix.loc[hour, station])
        assert value == 1, (hour, station, value)

    # oturumun kapsamadigi bir onceki ve bir sonraki saat varsa 1 olmak zorunda degil
    # ama beklenen listede olmayan komsu saatlerin bu oturumdan gelmedigini not ediyoruz
    print("matris beklenen saatlerde 1")
    print()


# kisa orta ve geceyarisini asan ornekler
sample_ids = [
    sessions.iloc[0]["sessionID"],
    sessions.iloc[len(sessions) // 2]["sessionID"],
]

# geceyarisini asan bir oturum bul
overnight = sessions[
    sessions["connectionTime_utc"].dt.normalize()
    != sessions["disconnectTime_utc"].dt.normalize()
].iloc[0]
sample_ids.append(overnight["sessionID"])

for sid in sample_ids:
    manual_check(sid)

print("elle kontrol tamam")

sessionID 2_39_78_362_2018-04-25 11:08:04.400812
stationID 2-39-78-362
connection 2018-04-25 11:08:04+00:00
disconnect 2018-04-25 13:20:10+00:00
beklenen dolu saat sayisi 2
beklenen ilk/son 2018-04-25 12:00:00+00:00 2018-04-25 13:00:00+00:00
matris beklenen saatlerde 1

sessionID 2_39_89_439_2019-01-15 17:01:54.516847
stationID 2-39-89-439
connection 2019-01-15 17:01:55+00:00
disconnect 2019-01-16 01:56:42+00:00
beklenen dolu saat sayisi 8
beklenen ilk/son 2019-01-15 18:00:00+00:00 2019-01-16 01:00:00+00:00
matris beklenen saatlerde 1

sessionID 2_39_95_27_2018-04-25 13:45:09.617470
stationID 2-39-95-27
connection 2018-04-25 13:45:10+00:00
disconnect 2018-04-26 00:56:16+00:00
beklenen dolu saat sayisi 11
beklenen ilk/son 2018-04-25 14:00:00+00:00 2018-04-26 00:00:00+00:00
matris beklenen saatlerde 1

elle kontrol tamam


## 6 ornek hesap aciklamasi

yapilanlar.mddeki ornek: 18:10 baglanip 21:05 ayrilirsa 19 20 21 dolu olur
asagida ayni kurali sentetik zamanla dogruluyorum

In [7]:
conn = pd.Timestamp("2020-01-15 18:10:00", tz="UTC")
disc = pd.Timestamp("2020-01-15 21:05:00", tz="UTC")
hours = occupied_hours_for_session(conn, disc)
print([h.strftime("%H:%M") for h in hours])
assert [h.hour for h in hours] == [19, 20, 21]
print("ornek kural dogru")

['19:00', '20:00', '21:00']
ornek kural dogru


## sonuc

- 14 dosya 31424 oturum 55 nokta
- eksik tarih hatali aralik tekrar id 0
- saatlik matris 29705 x 55
- elle secilen oturumlar ve 18:10-21:05 ornegi egenin kuraliyla uyumlu

sonraki adim kapsama ve 2020 bosluklari